In [1]:
import matplotlib.pyplot as plt
import datetime as dt
import pandas as pd
import numpy as np
import scipy as sp

from pathlib import Path


In [2]:
HARD_DRIVE_LOC = Path('/Volumes/Elements/UBNA_array_tests2026')
experiment_folder = HARD_DRIVE_LOC / 'recover-20260604'
all_wav_files = list(experiment_folder.glob('STF_*/2026*0.WAV'))
all_sync_wav_files = list(experiment_folder.glob('STF_*/2026*_SYNC.WAV'))
all_csv_files = list(experiment_folder.glob('STF_*/2026*.CSV'))
all_wav_files, all_sync_wav_files, all_csv_files

([PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_200000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_203000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_210000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_213000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_220000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_223000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_230000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260603_233000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260604_000000.WAV'),
  PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260604_003000.WAV'),


In [3]:
SD_CARD_TO_AUDIOMOTH_NUM = {'STF_026': '014', 'STF_028' : '045', 
                            'STF_057': '015', 'STF_109': '008', 
                            'STF_114': '040', 'STF_053': '032', 
                            'STF_021': '016', 'STF_116': '036', 
                            'STF_060': '004', 'STF_025': '042',  
                            'STF_027': '020',
                            'STF_113': '044', 'STF_030': '012'}
FREQ_UPPER_LIM = 6/8
SD_CARD_TO_AUDIOMOTH_NUM

{'STF_026': '014',
 'STF_028': '045',
 'STF_057': '015',
 'STF_109': '008',
 'STF_114': '040',
 'STF_053': '032',
 'STF_021': '016',
 'STF_116': '036',
 'STF_060': '004',
 'STF_025': '042',
 'STF_027': '020',
 'STF_113': '044',
 'STF_030': '012'}

In [4]:
AUDIOMOTH_AT_PATCHA = {'014':'A1 (top)', '045':'A1 (bottom)',
                       '015':'A4 (top)', '008':'A4 (bottom)',
                       '040':'A7 (top)', '032':'A7 (bottom)'}
AUDIOMOTH_AT_PATCHE = {'016':'E2 (top)', '036':'E2 (bottom)',
                       '004':'E4 (top)', '042':'E4 (bottom)',
                       '020':'E8 (alone)',
                       '044':'E9 (top)', '012':'E9 (bottom)'}

AUDIOMOTH_AT_ALLPATCHES = AUDIOMOTH_AT_PATCHA | AUDIOMOTH_AT_PATCHE

## Reproduce the AudioMoth GPS Sync algorithm in Python

This section implements the synchronization calculations used by the published
`audiomoth-utils` synchronizer (version 1.10.0) using only NumPy, pandas,
SciPy, and Python's standard library.

The important input columns are `AUDIOMOTH_TIME`, `TOTAL_SAMPLES`,
`TIMER_COUNT`, `BUFFERS_FILLED`, and `BUFFERS_WRITTEN`. The GPS RMC columns
are useful for diagnosis, but the official synchronizer does **not** use them
to position samples.

The output is a mono, 16-bit PCM WAV with a standard WAV header. Its samples
are GPS-synchronized, but this notebook implementation does not copy the
AudioMoth `LIST/INFO` or trailing GUANO metadata chunks. It also deliberately
stops on missing audio buffers rather than silently repairing a supposedly
valid input file.


In [5]:
valid_csv_files = []
for csv_file in all_csv_files:
    csv_file_dt = dt.datetime.strptime(csv_file.name, '%Y%m%d_%H%M%S.CSV')
    if csv_file_dt == dt.datetime(2026, 6, 4, 20, 30, 0):
        valid_csv_files += [csv_file]
valid_csv_files

[PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_025/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_116/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_027/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_030/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_060/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.CSV')]

In [6]:
valid_wav_files = []
for wav_file in all_wav_files:
    wave_file_dt = dt.datetime.strptime(wav_file.name, '%Y%m%d_%H%M%S.WAV')
    if wave_file_dt == dt.datetime(2026, 6, 4, 20, 30, 0):
        valid_wav_files += [wav_file]
valid_wav_files

[PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_021/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_025/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_116/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_027/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_030/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_060/20260604_203000.WAV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.WAV')]

### 1. Rebuild the file dictionaries without shared lists

`dict.fromkeys(keys, [])` gives every key the same mutable list. That made the
earlier `STF_027` expression return an `STF_021` file in the saved notebook.
The comprehensions below allocate a separate list for every SD card.


In [7]:
sync_csv_path = valid_csv_files[6]
raw_wav_path = valid_wav_files[6]

assert sync_csv_path.parent == raw_wav_path.parent
assert sync_csv_path.stem == raw_wav_path.stem, (
    "The WAV and CSV must have identical base names. "
    f"Got {raw_wav_path.name} and {sync_csv_path.name}."
)

sync_csv_path, raw_wav_path


(PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.CSV'),
 PosixPath('/Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.WAV'))

### 2. Constants and small helpers

These constants come from the AudioMoth firmware/synchronizer timing model.
All sample-to-PPS gaps below are measured in microseconds.


In [8]:
import math
import wave
import warnings

from scipy.io import wavfile

MICROSECONDS_PER_SECOND = 1_000_000.0
MILLISECONDS_PER_SECOND = 1_000.0

MAX_HFXO_ERROR_ABSOLUTE = 1000 / 1_000_000
MAX_HFXO_ERROR_RELATIVE = 40 / 1_000_000
MAX_LFXO_ERROR = 100 / 1_000_000

CLOCK_DIVIDER = 4
CONVERSION_CYCLES = 12
ACQUISITION_CYCLES = 16
CLOCK_FREQUENCY = 48_000_000
MAXIMUM_REFERENCE_SAMPLE_RATE = 384_000
MAXIMUM_ALLOWABLE_SAMPLE_RATE = 192_000

PPS_CLOCK_TICK_OFFSET = 2 + CLOCK_DIVIDER
MAXIMUM_ALLOWABLE_PPS_OFFSET = (
    PPS_CLOCK_TICK_OFFSET / CLOCK_FREQUENCY * MICROSECONDS_PER_SECOND
)

NUMBER_OF_FIRMWARE_BUFFERS = 8
FIRMWARE_BUFFER_SIZE_BYTES = 32 * 1024
BYTES_PER_SAMPLE = 2


def js_round(value):
    """Match JavaScript Math.round, including its behavior for negative halves."""
    return int(math.floor(float(value) + 0.5))


def calculate_interval_sample_rate(interval):
    """Infer the ADC sample rate between two accepted GPS PPS events."""
    usable_time_us = (
        interval["time_interval_seconds"] * MICROSECONDS_PER_SECOND
        - interval["first_sample_gap_us"]
        - interval["last_sample_gap_us"]
    )
    interval["sample_rate"] = (
        (interval["number_of_samples"] - 1)
        * MICROSECONDS_PER_SECOND
        / usable_time_us
    )
    return interval["sample_rate"]


### 3. Load and validate the raw WAV and synchronization CSV

SciPy memory-maps the raw WAV, so a long 192 kHz recording does not need to be
copied into RAM all at once.


In [9]:
required_columns = ["PPS_NUMBER", "AUDIOMOTH_TIME", "TOTAL_SAMPLES",
                    "TIMER_COUNT", "BUFFERS_FILLED", "BUFFERS_WRITTEN"]

sync_data = pd.read_csv(sync_csv_path)
missing_columns = sorted(set(required_columns) - set(sync_data.columns))
assert not missing_columns, f"CSV is missing required columns: {missing_columns}"
assert len(sync_data) >= 2, "The CSV must contain at least two PPS events."

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    source_sample_rate, raw_samples = wavfile.read(raw_wav_path, mmap=True)

assert raw_samples.ndim == 1, "The AudioMoth GPS Sync WAV must be mono."
assert raw_samples.dtype == np.int16, (f"Expected signed 16-bit PCM, got {raw_samples.dtype}.")

sync_data["audiomoth_datetime"] = pd.to_datetime(sync_data["AUDIOMOTH_TIME"], utc=True, errors="raise")
sync_data["audiomoth_time_ms"] = sync_data["audiomoth_datetime"].astype("int64") // 1_000_000

filename_datetime = pd.to_datetime(raw_wav_path.stem, format="%Y%m%d_%H%M%S", utc=True)
first_timestamp_difference_ms = abs((sync_data["audiomoth_datetime"].iloc[0] - filename_datetime).total_seconds() * 1000)
assert first_timestamp_difference_ms <= 500, (
    f"The first CSV AudioMoth timestamp differs from the WAV filename by {first_timestamp_difference_ms:.3f} ms.")

print(f"Raw WAV: {raw_wav_path}")
print(f"Sync CSV: {sync_csv_path}")
print(f"Nominal sample rate: {source_sample_rate:,} Hz")
print(f"Raw samples: {len(raw_samples):,}")
print(f"PPS rows: {len(sync_data):,}")


Raw WAV: /Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.WAV
Sync CSV: /Volumes/Elements/UBNA_array_tests2026/recover-20260604/STF_113/20260604_203000.CSV
Nominal sample rate: 192,000 Hz
Raw samples: 322,560,000
PPS rows: 1,630


### 4. Check for dropped firmware buffers

The desktop application can optionally replace missing buffers with zeros.
For this first valid-file experiment, stopping is safer: a buffer overflow
means the input is not the clean one-file example we intended to test.


In [10]:
buffer_overflows = []
previous_skipped_buffers = 0

for row_index in range(1, len(sync_data)):
    buffers_filled = int(sync_data["BUFFERS_FILLED"].iloc[row_index])
    buffers_written = int(sync_data["BUFFERS_WRITTEN"].iloc[row_index])
    difference = buffers_filled - buffers_written - previous_skipped_buffers

    if difference >= NUMBER_OF_FIRMWARE_BUFFERS:
        skipped = NUMBER_OF_FIRMWARE_BUFFERS * (
            difference // NUMBER_OF_FIRMWARE_BUFFERS
        )
        missing_samples = (
            FIRMWARE_BUFFER_SIZE_BYTES // BYTES_PER_SAMPLE * skipped
        )
        previous_skipped_buffers += skipped
        buffer_overflows.append(
            {
                "csv_row": row_index,
                "skipped_buffers": skipped,
                "missing_samples": missing_samples,
            }
        )

assert not buffer_overflows, (
    "This file contains missing WAV buffers. Choose another valid pair or "
    "implement the application's resolve-WAV zero insertion option. "
    f"Events: {buffer_overflows[:5]}"
)
print("No dropped firmware buffers detected.")


No dropped firmware buffers detected.


### 5. Convert PPS rows into accepted timing intervals

The acceptance bounds model both AudioMoth oscillators. A bad PPS row can be
skipped by setting `RESOLVE_GPS = True`, matching the application's
“resolve GPS issues” option. This recording needs that option because it
contains a valid seven-second interval between accepted PPS anchors.


In [11]:
RESOLVE_GPS = True

sample_interval_us = MICROSECONDS_PER_SECOND / source_sample_rate
oversample_rate = 2 ** math.floor(
    math.log2(MAXIMUM_REFERENCE_SAMPLE_RATE / source_sample_rate)
)
clock_ticks_between_samples = CLOCK_FREQUENCY / source_sample_rate
clock_ticks_to_complete_sample = (
    2
    + CLOCK_DIVIDER
    * (2 + oversample_rate * (ACQUISITION_CYCLES + CONVERSION_CYCLES))
)

timer_count = sync_data["TIMER_COUNT"].to_numpy(dtype=np.float64)
time_to_next_sample_us = np.where(
    timer_count <= clock_ticks_to_complete_sample,
    clock_ticks_to_complete_sample - timer_count,
    clock_ticks_between_samples + clock_ticks_to_complete_sample - timer_count,
) / CLOCK_FREQUENCY * MICROSECONDS_PER_SECOND

total_samples_at_pps = sync_data["TOTAL_SAMPLES"].to_numpy(dtype=np.int64)
time_at_pps_ms = sync_data["audiomoth_time_ms"].to_numpy(dtype=np.int64)

intervals = []
rejected_pps_rows = []
cumulative_time_seconds = 0
current_index = 0
current_total_samples = int(total_samples_at_pps[0])
current_time_ms = int(time_at_pps_ms[0])
sample_rate_total = 0
sample_rate_count = 0

for next_index in range(1, len(sync_data)):
    next_total_samples = int(total_samples_at_pps[next_index])
    next_time_ms = int(time_at_pps_ms[next_index])

    number_of_samples = next_total_samples - current_total_samples
    measured_interval_ms = next_time_ms - current_time_ms
    rounded_interval_seconds = js_round(
        measured_interval_ms / MILLISECONDS_PER_SECOND
    )

    target_rate = (
        source_sample_rate
        if sample_rate_count == 0
        else sample_rate_total / sample_rate_count
    )
    maximum_time_error_ms = math.ceil(
        MAX_LFXO_ERROR
        * rounded_interval_seconds
        * MILLISECONDS_PER_SECOND
    )
    maximum_sample_error = math.ceil(
        (
            MAX_HFXO_ERROR_ABSOLUTE
            if sample_rate_count == 0
            else MAX_HFXO_ERROR_RELATIVE
        )
        * (source_sample_rate if sample_rate_count == 0 else target_rate)
        * rounded_interval_seconds
    )

    time_error_ms = abs(measured_interval_ms - rounded_interval_seconds * MILLISECONDS_PER_SECOND)
    sample_error = abs(number_of_samples - rounded_interval_seconds * target_rate)
    valid_interval = rounded_interval_seconds > 0 and time_error_ms <= maximum_time_error_ms and sample_error <= maximum_sample_error

    if not valid_interval:
        rejected_pps_rows.append(next_index)
        if not RESOLVE_GPS:
            raise ValueError(
                "Misaligned PPS event at CSV row "
                f"{next_index}: {number_of_samples} samples over {measured_interval_ms:.3f} ms; "
                f"time error {time_error_ms:.6f}/{maximum_time_error_ms} ms, "
                f"sample error {sample_error:.3f}/{maximum_sample_error} samples."
            )
        continue

    if rounded_interval_seconds > 1 and not RESOLVE_GPS:
        raise ValueError(
            f"Missing PPS event before CSV row {next_index}; "
            f"accepted interval is {rounded_interval_seconds} seconds."
        )

    sample_rate_total += number_of_samples
    sample_rate_count += rounded_interval_seconds

    interval = {
        "interval_index": len(intervals),
        "start_pps_index": current_index,
        "end_pps_index": next_index,
        "time_interval_seconds": rounded_interval_seconds,
        "cumulative_time_seconds": cumulative_time_seconds,
        "number_of_samples": number_of_samples,
        "first_sample_gap_us": float(time_to_next_sample_us[current_index]),
        "last_sample_gap_us": float(
            sample_interval_us - time_to_next_sample_us[next_index]
        ),
        "extrapolated": False,
    }
    calculate_interval_sample_rate(interval)
    intervals.append(interval)

    cumulative_time_seconds += rounded_interval_seconds
    current_index = next_index
    current_total_samples = next_total_samples
    current_time_ms = next_time_ms

assert intervals, "No valid interval exists between PPS events."

average_sample_rate = (
    sum(x["number_of_samples"] for x in intervals)
    / sum(x["time_interval_seconds"] for x in intervals)
)

print(f"Accepted intervals: {len(intervals):,}")
print(f"Rejected PPS rows: {rejected_pps_rows}")
print(f"Average raw rate: {average_sample_rate:.6f} samples/s")


Accepted intervals: 1,629
Rejected PPS rows: []
Average raw rate: 191964.657738 samples/s


### 6. Apply timing corrections and repair an unanchored recording tail

These small corrections handle a sample occurring extremely close to a PPS
edge and align each value to the midpoint of the ADC acquisition window.
If real PPS anchors stop before second 1,680, one clearly marked interval is
then extrapolated from the final 300 anchored seconds of sample-clock data.


In [12]:
# Correct PPS/sample ordering ambiguities.
for interval_index in range(len(intervals) - 1):
    interval = intervals[interval_index]
    next_interval = intervals[interval_index + 1]

    if (
        interval["last_sample_gap_us"] < MAXIMUM_ALLOWABLE_PPS_OFFSET
        and js_round(interval["sample_rate"] - average_sample_rate) == -1
        and js_round(next_interval["sample_rate"] - average_sample_rate) == 1
    ):
        interval["last_sample_gap_us"] = sample_interval_us
        calculate_interval_sample_rate(interval)
        next_interval["first_sample_gap_us"] = 0.0
        calculate_interval_sample_rate(next_interval)

if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
    first_interval = intervals[0]
    if js_round(first_interval["sample_rate"] - average_sample_rate) == 1:
        first_interval["first_sample_gap_us"] -= sample_interval_us
        calculate_interval_sample_rate(first_interval)

    for interval_index in range(len(intervals) - 1):
        interval = intervals[interval_index]
        next_interval = intervals[interval_index + 1]
        if (
            interval["last_sample_gap_us"] < MAXIMUM_ALLOWABLE_PPS_OFFSET
            and js_round(interval["sample_rate"] - average_sample_rate) == -1
            and js_round(next_interval["sample_rate"] - average_sample_rate) == 0
        ):
            interval["last_sample_gap_us"] = sample_interval_us
            calculate_interval_sample_rate(interval)
            next_interval["first_sample_gap_us"] = sample_interval_us
            calculate_interval_sample_rate(next_interval)

    for interval in intervals:
        if js_round(interval["sample_rate"] - average_sample_rate) == -1:
            interval["first_sample_gap_us"] += sample_interval_us
            calculate_interval_sample_rate(interval)

first_sample_is_before_first_interval = (
    intervals[0]["first_sample_gap_us"] < 0
)

# Align samples to the midpoint of the ADC acquisition period.
last_acquisition_ends = 1 + CLOCK_DIVIDER * (CONVERSION_CYCLES + 1)
first_acquisition_starts = (
    clock_ticks_to_complete_sample - 1 - CLOCK_DIVIDER
)
adc_time_offset_us = (
    MICROSECONDS_PER_SECOND
    * (last_acquisition_ends + first_acquisition_starts)
    / 2
    / CLOCK_FREQUENCY
)

for interval_index, interval in enumerate(intervals):
    interval["first_sample_gap_us"] -= adc_time_offset_us
    interval["last_sample_gap_us"] += adc_time_offset_us

    if interval["first_sample_gap_us"] < 0:
        interval["number_of_samples"] -= 1
        interval["first_sample_gap_us"] += sample_interval_us
        if interval_index == 0:
            first_sample_is_before_first_interval = True
        else:
            previous_interval = intervals[interval_index - 1]
            previous_interval["number_of_samples"] += 1
            previous_interval["last_sample_gap_us"] = (
                sample_interval_us - interval["first_sample_gap_us"]
            )

for interval in intervals:
    calculate_interval_sample_rate(interval)

unusual_intervals = [
    interval["interval_index"]
    for interval in intervals
    if js_round(interval["sample_rate"] - average_sample_rate) != 0
]
if unusual_intervals and not RESOLVE_GPS:
    raise ValueError(
        "Unusual sample count remains in intervals "
        f"{unusual_intervals[:20]}."
    )

# The firmware's 192 kHz mode misses the first sample at each interval.
if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
    for interval in intervals:
        interval["first_sample_gap_us"] += sample_interval_us
        calculate_interval_sample_rate(interval)

# Repair a final section for which recording samples exist but PPS anchors do not.
TARGET_OUTPUT_DURATION_SECONDS = 1680
TAIL_RATE_ESTIMATION_SECONDS = 300
anchored_duration_seconds = sum(interval["time_interval_seconds"] for interval in intervals)
seconds_to_extrapolate = TARGET_OUTPUT_DURATION_SECONDS - anchored_duration_seconds

if seconds_to_extrapolate < 0:
    raise ValueError(f"The accepted GPS intervals already exceed the requested {TARGET_OUTPUT_DURATION_SECONDS}-second output.")

if seconds_to_extrapolate > 0:
    recent_intervals = []
    recent_duration_seconds = 0
    for interval in reversed(intervals):
        recent_intervals.append(interval)
        recent_duration_seconds += interval["time_interval_seconds"]
        if recent_duration_seconds >= TAIL_RATE_ESTIMATION_SECONDS:
            break

    estimated_tail_rate = sum(interval["number_of_samples"] for interval in recent_intervals) / sum(interval["time_interval_seconds"] for interval in recent_intervals)
    extrapolated_sample_count = js_round(seconds_to_extrapolate * estimated_tail_rate)
    extrapolated_first_gap_us = float(time_to_next_sample_us[current_index])
    extrapolated_last_gap_us = seconds_to_extrapolate * MICROSECONDS_PER_SECOND - extrapolated_first_gap_us - (extrapolated_sample_count - 1) * MICROSECONDS_PER_SECOND / estimated_tail_rate

    # Apply the same ADC midpoint alignment used for measured intervals.
    extrapolated_first_gap_us -= adc_time_offset_us
    extrapolated_last_gap_us += adc_time_offset_us
    if extrapolated_first_gap_us < 0:
        extrapolated_sample_count -= 1
        extrapolated_first_gap_us += sample_interval_us
        intervals[-1]["number_of_samples"] += 1
        intervals[-1]["last_sample_gap_us"] = sample_interval_us - extrapolated_first_gap_us
        calculate_interval_sample_rate(intervals[-1])

    if source_sample_rate == MAXIMUM_ALLOWABLE_SAMPLE_RATE:
        extrapolated_first_gap_us += sample_interval_us

    extrapolated_interval = {
        "interval_index": len(intervals),
        "start_pps_index": current_index,
        "end_pps_index": None,
        "time_interval_seconds": seconds_to_extrapolate,
        "cumulative_time_seconds": anchored_duration_seconds,
        "number_of_samples": extrapolated_sample_count,
        "first_sample_gap_us": extrapolated_first_gap_us,
        "last_sample_gap_us": extrapolated_last_gap_us,
        "extrapolated": True,
    }
    calculate_interval_sample_rate(extrapolated_interval)
    intervals.append(extrapolated_interval)

raw_samples_used = sum(interval["number_of_samples"] for interval in intervals)
discarded_raw_samples = len(raw_samples) - raw_samples_used
assert discarded_raw_samples >= 0, f"The repaired intervals require {-discarded_raw_samples:,} more raw samples than the WAV contains."
assert sum(interval["time_interval_seconds"] for interval in intervals) == TARGET_OUTPUT_DURATION_SECONDS

if seconds_to_extrapolate > 0:
    print(f"Extrapolated final {seconds_to_extrapolate} seconds at {estimated_tail_rate:.6f} raw samples/s")
    print(f"Raw samples intentionally discarded after scheduled endpoint: {discarded_raw_samples:,} ({discarded_raw_samples / estimated_tail_rate:.6f} s)")

interval_table = pd.DataFrame(intervals)
interval_columns = [
        "interval_index",
        "start_pps_index",
        "end_pps_index",
        "time_interval_seconds",
        "number_of_samples",
        "sample_rate",
        "first_sample_gap_us",
        "last_sample_gap_us",
        "extrapolated",
]
pd.concat([interval_table[interval_columns].head(5), interval_table[interval_columns].tail(1)])


,interval_index,start_pps_index,end_pps_index,time_interval_seconds,number_of_samples,sample_rate,first_sample_gap_us,last_sample_gap_us,extrapolated
0,0,0,1,1,191965,191965.635707,5.833333,2.687500,False
1,1,1,2,1,191965,191965.635707,7.729167,0.791667,False
2,2,2,3,1,191964,191965.639528,9.625000,4.125000,False
3,3,3,4,1,191965,191965.635707,6.291667,2.229167,False
4,4,4,5,1,191965,191965.639707,8.187500,0.354167,False
1628,1628,1628,1629,1,191964,191965.583537,8.854167,4.604167,False


### 7. Resample every accepted PPS interval onto the GPS time grid

The official app offers linear interpolation and nearest-neighbour sampling.
Linear interpolation is its first/default option and is used here. Processing
one PPS interval at a time keeps RAM use modest even for a long recording.


In [13]:
def round_and_clip_int16(values):
    """Match Math.round and convert interpolated values to 16-bit PCM."""
    rounded = np.floor(values + 0.5)
    return np.clip(rounded, -32768, 32767).astype("<i2")


def resample_one_interval(
    samples,
    first_input_index,
    interval,
    target_sample_rate,
    algorithm="linear",
):
    """Return one GPS-aligned interval and the next raw sample index."""
    input_count = int(interval["number_of_samples"])
    final_input_index = first_input_index + input_count
    if final_input_index >= len(samples):
        raise ValueError(
            "The CSV asks for more samples than are present in the raw WAV: "
            f"needed index {final_input_index:,}, "
            f"last available index is {len(samples) - 1:,}."
        )

    if first_input_index == 0:
        # The official stream starts with previous == next == sample 0.
        # Duplicate that value so it has both the previous- and next-sample
        # timestamps during the first interval.
        input_indices = np.concatenate(
            (
                np.array([0], dtype=np.int64),
                np.arange(0, final_input_index + 1, dtype=np.int64),
            )
        )
    else:
        input_indices = np.arange(
            first_input_index - 1,
            final_input_index + 1,
            dtype=np.int64,
        )
    input_values = np.asarray(samples[input_indices], dtype=np.float64)

    first_gap_seconds = (
        interval["first_sample_gap_us"] / MICROSECONDS_PER_SECOND
    )
    previous_gap_seconds = first_gap_seconds - (
        sample_interval_us / MICROSECONDS_PER_SECOND
    )
    input_times = np.empty(len(input_indices), dtype=np.float64)
    input_times[0] = previous_gap_seconds
    input_times[1:] = first_gap_seconds + (
        np.arange(len(input_indices) - 1, dtype=np.float64)
        / interval["sample_rate"]
    )

    number_of_output_samples = (
        int(interval["time_interval_seconds"]) * target_sample_rate
    )
    output_times = (
        np.arange(number_of_output_samples, dtype=np.float64)
        / target_sample_rate
    )

    if algorithm == "linear":
        output_values = np.interp(output_times, input_times, input_values)
    elif algorithm == "nearest":
        right = np.searchsorted(input_times, output_times, side="left")
        right = np.clip(right, 1, len(input_times) - 1)
        left = right - 1
        choose_right = (
            input_times[right] - output_times
            < 0.5 / interval["sample_rate"]
        )
        selected = np.where(choose_right, right, left)
        output_values = input_values[selected]
    else:
        raise ValueError("algorithm must be 'linear' or 'nearest'")

    return round_and_clip_int16(output_values), final_input_index


def write_gps_synchronized_wav(
    samples,
    intervals,
    source_rate,
    output_path,
    target_rate=None,
    algorithm="linear",
):
    """Write a mono 16-bit PCM WAV on an exact GPS-spaced sample grid."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    target_rate = source_rate if target_rate is None else int(target_rate)

    if target_rate < source_rate:
        raise ValueError(
            "The AudioMoth sync app does not permit a target rate below "
            "the raw WAV sample rate."
        )

    first_input_index = 1 if first_sample_is_before_first_interval else 0
    total_output_samples = 0

    with wave.open(str(output_path), "wb") as output_wav:
        output_wav.setnchannels(1)
        output_wav.setsampwidth(BYTES_PER_SAMPLE)
        output_wav.setframerate(target_rate)

        for interval_index, interval in enumerate(intervals):
            aligned_samples, first_input_index = resample_one_interval(
                samples=samples,
                first_input_index=first_input_index,
                interval=interval,
                target_sample_rate=target_rate,
                algorithm=algorithm,
            )
            output_wav.writeframesraw(aligned_samples.tobytes())
            total_output_samples += len(aligned_samples)

            if (
                (interval_index + 1) % 60 == 0
                or interval_index + 1 == len(intervals)
            ):
                print(
                    f"Processed {interval_index + 1:,}/{len(intervals):,} "
                    "PPS intervals"
                )

    expected_output_samples = (
        sum(x["time_interval_seconds"] for x in intervals) * target_rate
    )
    assert total_output_samples == expected_output_samples
    return output_path


### 8. Produce the synchronized WAV

The result is written beside the raw recording with `_PYTHON_SYNC.WAV` so it
cannot overwrite the official `_SYNC.WAV`. For the selected pair, this is
`STF_027/20260604_203000_PYTHON_SYNC.WAV`.


In [14]:
python_sync_wav_path = raw_wav_path.with_name(
    f"{raw_wav_path.stem}_PYTHON_SYNC.WAV"
)

python_sync_wav_path = write_gps_synchronized_wav(
    samples=raw_samples,
    intervals=intervals,
    source_rate=source_sample_rate,
    output_path=python_sync_wav_path,
    target_rate=source_sample_rate,
    algorithm="linear",
)

print(f"Wrote: {python_sync_wav_path}")
print(
    "GPS duration: "
    f"{sum(x['time_interval_seconds'] for x in intervals):,.0f} seconds"
)


Processed 60/1,629 PPS intervals
Processed 120/1,629 PPS intervals
Processed 180/1,629 PPS intervals
Processed 240/1,629 PPS intervals
Processed 300/1,629 PPS intervals
Processed 360/1,629 PPS intervals
Processed 420/1,629 PPS intervals
Processed 480/1,629 PPS intervals
Processed 540/1,629 PPS intervals
Processed 600/1,629 PPS intervals
Processed 660/1,629 PPS intervals
Processed 720/1,629 PPS intervals
Processed 780/1,629 PPS intervals
Processed 840/1,629 PPS intervals
Processed 900/1,629 PPS intervals
Processed 960/1,629 PPS intervals
Processed 1,020/1,629 PPS intervals
Processed 1,080/1,629 PPS intervals
Processed 1,140/1,629 PPS intervals
Processed 1,200/1,629 PPS intervals
Processed 1,260/1,629 PPS intervals
Processed 1,320/1,629 PPS intervals
Processed 1,380/1,629 PPS intervals
Processed 1,440/1,629 PPS intervals
Processed 1,500/1,629 PPS intervals
Processed 1,560/1,629 PPS intervals
Processed 1,620/1,629 PPS intervals
Processed 1,629/1,629 PPS intervals
Wrote: /Volumes/Elements/

### 9. Optional comparison with the official application output

This compares audio samples in chunks, without loading either complete output
into RAM. WAV headers can differ because the Python file intentionally omits
AudioMoth metadata.


In [15]:
official_sync_wav_path = raw_wav_path.with_name(
    f"{raw_wav_path.stem}_SYNC.WAV"
)

if official_sync_wav_path.exists():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        official_rate, official_samples = wavfile.read(
            official_sync_wav_path, mmap=True
        )
        python_rate, python_samples = wavfile.read(
            python_sync_wav_path, mmap=True
        )

    assert official_rate == python_rate
    common_sample_count = min(len(official_samples), len(python_samples))
    comparison_chunk_size = 5_000_000
    mismatch_count = 0
    maximum_absolute_difference = 0

    for start in range(0, common_sample_count, comparison_chunk_size):
        stop = min(start + comparison_chunk_size, common_sample_count)
        difference = (
            np.asarray(python_samples[start:stop], dtype=np.int32)
            - np.asarray(official_samples[start:stop], dtype=np.int32)
        )
        mismatch_count += np.count_nonzero(difference)
        if len(difference):
            maximum_absolute_difference = max(
                maximum_absolute_difference,
                int(np.max(np.abs(difference))),
            )

    comparison = pd.Series(
        {
            "official_samples": len(official_samples),
            "python_samples": len(python_samples),
            "common_samples": common_sample_count,
            "mismatched_common_samples": mismatch_count,
            "maximum_absolute_difference": maximum_absolute_difference,
        }
    )
    display(comparison)
else:
    print(f"No official comparison file found at {official_sync_wav_path}")


official_samples               322560000
python_samples                 322560000
common_samples                 322560000
mismatched_common_samples          10569
maximum_absolute_difference          495
dtype: int64